### Setup

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")

MODEL = "gpt-4.1-mini"

In [ ]:
# Absolute path is important: the server needs a full path for its allowed directory.
SANDBOX_DIR = os.path.abspath("secret_project_007")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Drop in a couple of sample files for the agent to find.
# encoding="utf-8" keeps the em dashes (—) safe on Windows (which defaults to cp1252).
with open(os.path.join(SANDBOX_DIR, "notes.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Project Falcon — kickoff notes\n"
        "- Estimated budget: $42,000\n"
        "- Launch target: end of Q3\n"
        "- Owner: Priya\n"
        "- Core team: Marco (engineering), Lena (design), Sam (go-to-market)\n"
        "What it is:\n"
        "Falcon is a tech news and discussion community with AI avatars who provide commentary\n"
        "It's like a Hacker News + a Late Night Talk Show, but in written format and with AI commentators.\n"
        "The bet: engaging AI avatars will quickly become fan-favourites because of the unique dynamic between them.\n"
        "Premise:\n"
        "Ship before the competition announces theirs; quiet until then\n"
        "Hacker News is the benchmark — we study it constantly to understand what the community wants\n"
        "Two competitors rumored to be building something similar — speed is the moat\n"
        "'Falcon' is a placeholder; marketing hates it, but it has stuck\n"
        "Priya's one hard rule: do not slip the Q3 date to add features\n"
        "Open question: what are the most popular categories on Hacker News?\n"
    )

with open(os.path.join(SANDBOX_DIR, "todo.md"), "w", encoding="utf-8") as f:
    f.write(
        "# To do\n"
        "- [X] Book venue\n"
        "- [ ] Confirm budget with finance\n"
        "- [X] Lock the core team\n"
        "- [X] Brief the team on keeping this quiet\n"
        "- [ ] Draft announcement\n"
        "- [ ] Decide whether to keep the 'Falcon' name or rebrand before launch\n"
        "- [ ] Pressure-test the Q3 timeline with engineering\n"
        "- [ ] Scan the Hacker News front page, group top 20 stories into broad categories\n"
    )

print(f"✅ Sandbox ready at: {SANDBOX_DIR}")
print("   Files:", os.listdir(SANDBOX_DIR))

#==================================================

# Create a decoy folder
OTHER_DIR = os.path.abspath("secret_project_006")
os.makedirs(OTHER_DIR, exist_ok=True)

# Feels empty.. Drop in a decoy file.
with open(os.path.join(OTHER_DIR, "decoy_file.txt"), "w") as f:
    f.write("This is a decoy file")

print(f"✅ Decoy folder is ready at: {OTHER_DIR}")
print("   Files:", os.listdir(OTHER_DIR))

### Check that Node.js + npx are available

In [ ]:
# If this errors, install Node from https://nodejs.org/ and ensure it's in your PATH.
!node --version
!npx --version

## Filesystem MCP Server

In [ ]:
filesystem_server_params = {
    "command": "npx",
    "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        SANDBOX_DIR
    ]
}

In [ ]:
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    fs_tools = await server2.list_tools()

    print(f"✅ MCP Server connected. Server offers {len(fs_tools)} tools:\n")
    for tool in fs_tools:
        print(f"🔧 - {tool.name}: {tool.description.strip().splitlines()[0]}")
    

## Filesystem Agent (with MCP Server)

In [ ]:
FILES_AGENT_PROMPT = f"""
You are a file assistant. You work inside the directory {SANDBOX_DIR}.
Always use FULL paths under {SANDBOX_DIR} (e.g. {SANDBOX_DIR}/notes.txt).
You have filesystem tools (via an MCP server) to list, read, search, write, and edit files.
When asked about files, first list or read what's there, then act based on what you actually
find. Be concise, and tell the user exactly which files you read or changed.
"""

In [ ]:
# Read Files
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"What files are in my folder, and what is each one about? Give me a one-line summary per file.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

In [ ]:
# Search files for a keyword
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params, client_session_timeout_seconds=60) as server2:
    files_agent = Agent(
        name="Files Agent",
        instructions=FILES_AGENT_PROMPT,
        model=MODEL,
        mcp_servers=[server2]
    )

    result = await Runner.run(
        files_agent,
        input=f"Search inside the files in the folder for keyword 'Budget', list the files that mention it and what they say about it.",
        max_turns=10
    )

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)